<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [3]</a>'.</span>

# Chapter 11: Scoring, Validation & Explanations

End-to-end scoring pipeline with holdout validation, model comparison, adversarial
validation, SHAP explanations, and error analysis.

**Sections:**
1. Run Scoring
2. Summary Metrics
3. Model Comparison Grid
4. Adversarial Pipeline Validation
5. Transformation Validation
6. Model Explanations (SHAP)
7. Customer Browser
8. Error Analysis
9. Export Results

In [ ]:
from customer_retention.analysis.notebook_progress import track_and_export_previous
track_and_export_previous("11_scoring_validation.ipynb")

import sys
from pathlib import Path

from customer_retention.core.config.experiments import EXPERIMENTS_DIR, FINDINGS_DIR

In [ ]:
# Discover the generated pipeline directory
generated_dir = Path("../generated_pipelines/local")
pipeline_dirs = sorted(generated_dir.glob("*/config.py"))
if not pipeline_dirs:
    raise FileNotFoundError(
        f"No generated pipeline found under {generated_dir}. Run notebook 10 first."
    )
PIPELINE_DIR = pipeline_dirs[-1].parent
sys.path.insert(0, str(PIPELINE_DIR))

from config import (
    PIPELINE_NAME, TARGET_COLUMN, RECOMMENDATIONS_HASH, MLFLOW_TRACKING_URI,
    FEAST_REPO_PATH, FEAST_FEATURE_VIEW, FEAST_ENTITY_KEY, FEAST_TIMESTAMP_COL,
    PRODUCTION_DIR, EXPERIMENTS_DIR as GEN_EXPERIMENTS_DIR,
    get_feast_data_path, get_gold_path, ARTIFACTS_PATH,
)

print(f"Pipeline: {PIPELINE_NAME}")
print(f"Pipeline dir: {PIPELINE_DIR}")
print(f"Experiments dir: {GEN_EXPERIMENTS_DIR}")
print(f"Recommendations hash: {RECOMMENDATIONS_HASH}")

## 11.1 Run Scoring

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import xgboost as xgb
from feast import FeatureStore
from customer_retention.transforms import TransformExecutor, ArtifactStore
from customer_retention.generators.pipeline_generator.models import (
    PipelineTransformationType, TransformationStep,
)
from config import EXCLUDED_SOURCES

_registry = ArtifactStore.from_manifest(Path(ARTIFACTS_PATH) / "manifest.yaml")
_executor = TransformExecutor()

# Import encoding/scaling steps from gold module
sys.path.insert(0, str(PIPELINE_DIR / "gold"))
from gold_features import ENCODINGS, SCALINGS

ORIGINAL_COLUMN = f"original_{TARGET_COLUMN}"
PREDICTIONS_PATH = PRODUCTION_DIR / "data" / "scoring" / "predictions.parquet"

# Set tracking URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)


# --- Load holdout (from gold path which retains original_target, not Feast which excludes it) ---
features_df = pd.read_parquet(get_gold_path())

if ORIGINAL_COLUMN not in features_df.columns:
    raise ValueError(
        f"No holdout found (column '{ORIGINAL_COLUMN}' missing). "
        "Holdout must be created in silver layer BEFORE gold layer feature computation."
    )

scoring_mask = features_df[TARGET_COLUMN].isna() & features_df[ORIGINAL_COLUMN].notna()
scoring_df = features_df[scoring_mask].copy()
print(f"Found {len(scoring_df):,} holdout records for scoring")


# --- Feast features (fallback to parquet) ---
feast_path = Path(FEAST_REPO_PATH)
if (feast_path / "feature_store.yaml").exists():
    try:
        store = FeatureStore(repo_path=str(feast_path))
        entity_df = scoring_df[[FEAST_ENTITY_KEY, FEAST_TIMESTAMP_COL]].copy()
        exclude_cols = {FEAST_ENTITY_KEY, FEAST_TIMESTAMP_COL, TARGET_COLUMN, ORIGINAL_COLUMN}
        feature_cols = [
            c for c in scoring_df.columns
            if c not in exclude_cols and not c.startswith("original_")
        ]
        feature_refs = [f"{FEAST_FEATURE_VIEW}:{col}" for col in feature_cols]
        result_df = store.get_online_features(
            features=feature_refs,
            entity_rows=[{FEAST_ENTITY_KEY: eid} for eid in scoring_df[FEAST_ENTITY_KEY]]
        ).to_df()
        result_df[ORIGINAL_COLUMN] = scoring_df[ORIGINAL_COLUMN].values
        result_df[FEAST_ENTITY_KEY] = scoring_df[FEAST_ENTITY_KEY].values
        scoring_features = result_df
        print("Loaded features from Feast")
    except Exception as e:
        print(f"Feast retrieval failed ({e}), using parquet")
        scoring_features = scoring_df
else:
    print("Feast not initialized, using parquet directly")
    scoring_features = scoring_df


# --- Load best model ---
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(PIPELINE_NAME)
if not experiment:
    raise ValueError(f"Experiment {PIPELINE_NAME} not found")


def _find_best_parent_run(client, experiment_id):
    runs = client.search_runs(
        experiment_ids=[experiment_id],
        filter_string=f"tags.recommendations_hash = '{RECOMMENDATIONS_HASH}'",
        order_by=["metrics.best_roc_auc DESC"], max_results=1,
    )
    if not runs:
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            order_by=["metrics.best_roc_auc DESC"], max_results=1,
        )
    if not runs:
        raise ValueError("No runs found")
    return runs[0]


parent_run = _find_best_parent_run(client, experiment.experiment_id)
best_model_tag = parent_run.data.tags.get("best_model", "random_forest")
model_name = f"model_{best_model_tag}"
if RECOMMENDATIONS_HASH:
    model_name = f"{model_name}_{RECOMMENDATIONS_HASH}"

child_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'",
)
model_run = next(
    (c for c in child_runs if c.info.run_name == best_model_tag), parent_run
)
model_uri = f"runs:/{model_run.info.run_id}/{model_name}"
print(f"Loading model: {model_uri}")
loader = mlflow.xgboost if best_model_tag == "xgboost" else mlflow.sklearn
model = loader.load_model(model_uri)


# --- Prepare features (TransformExecutor, NOT LabelEncoder) ---
def prepare_features(df):
    df = df.copy()
    drop_cols = [FEAST_ENTITY_KEY, FEAST_TIMESTAMP_COL, ORIGINAL_COLUMN, TARGET_COLUMN]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
    df = df.drop(columns=[c for c in df.columns if c.startswith("original_")], errors="ignore")
    df = _executor.apply_all(df, ENCODINGS + SCALINGS, fit_mode=False, artifact_store=_registry)
    return df.select_dtypes(include=["int64", "float64", "int32", "float32"]).fillna(0)


X = prepare_features(scoring_features)
y_true = scoring_features[ORIGINAL_COLUMN].values

# --- Predict ---
print("Generating predictions...")
if hasattr(model, "predict_proba"):
    y_proba = model.predict_proba(X)[:, 1]
else:
    y_proba = model.predict(xgb.DMatrix(X, feature_names=list(X.columns)))
y_pred = (y_proba >= 0.5).astype(int)

# --- Metrics ---
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0,
}
print("\nValidation Metrics (vs original values):")
for name, value in metrics.items():
    print(f"  {name}: {value:.4f}")

# --- Save predictions ---
predictions_df = pd.DataFrame({
    FEAST_ENTITY_KEY: scoring_df[FEAST_ENTITY_KEY].values,
    "prediction": y_pred,
    "probability": y_proba,
    "actual": y_true,
    "correct": (y_pred == y_true).astype(int),
})
PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
predictions_df.to_parquet(PREDICTIONS_PATH, index=False)
print(f"\nPredictions saved: {PREDICTIONS_PATH}")
print(f"Correct: {predictions_df['correct'].sum():,}/{len(predictions_df):,} ({predictions_df['correct'].mean():.1%})")

## 11.2 Summary Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
)

y_true = predictions_df["actual"]
y_pred = predictions_df["prediction"]
y_proba = predictions_df["probability"]

metrics = {
    "Accuracy": accuracy_score(y_true, y_pred),
    "Precision": precision_score(y_true, y_pred, zero_division=0),
    "Recall": recall_score(y_true, y_pred, zero_division=0),
    "F1 Score": f1_score(y_true, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.0,
}

print("\n=== Scoring Validation Metrics ===")
for name, value in metrics.items():
    print(f"  {name}: {value:.4f}")

cm = confusion_matrix(y_true, y_pred)
print(f"\nConfusion Matrix:")
print(f"  TN={cm[0,0]:,}  FP={cm[0,1]:,}")
print(f"  FN={cm[1,0]:,}  TP={cm[1,1]:,}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_proba)
axes[0].plot(fpr, tpr, "b-", lw=2, label=f"ROC (AUC={metrics['ROC-AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve")
axes[0].legend()

# Probability distribution
axes[1].hist(y_proba[y_true == 0], bins=30, alpha=0.5, label="Actual=0", color="blue")
axes[1].hist(y_proba[y_true == 1], bins=30, alpha=0.5, label="Actual=1", color="red")
axes[1].axvline(x=0.5, color="black", linestyle="--", label="Threshold")
axes[1].set_xlabel("Predicted Probability")
axes[1].set_ylabel("Count")
axes[1].set_title("Probability Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## 11.3 Model Comparison Grid

Compare all trained models (Logistic Regression, Random Forest, XGBoost) on the holdout set.

**Grid Layout:**
- **Row 1**: Confusion matrices (counts and percentages)
- **Row 2**: ROC curves with AUC scores
- **Row 3**: Precision-Recall curves with PR-AUC scores

In [ ]:
from sklearn.metrics import (
    roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score,
)
from IPython.display import display, HTML

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(PIPELINE_NAME)

# Prepare features for scoring using TransformExecutor (NOT LabelEncoder)
X_holdout = prepare_features(scoring_features)
y_actual = predictions_df["actual"].values

# Get all logged models
logged_models = client.search_logged_models(experiment_ids=[experiment.experiment_id])

# Load all 3 model types
model_types = ["logistic_regression", "random_forest", "xgboost"]
model_display_names = ["Logistic Regression", "Random Forest", "XGBoost"]
loaded_models = {}
model_predictions = {}

for model_type, display_name in zip(model_types, model_display_names):
    model_name_pattern = f"model_{model_type}"
    if RECOMMENDATIONS_HASH:
        model_name_pattern = f"{model_name_pattern}_{RECOMMENDATIONS_HASH}"

    matching_model = None
    for lm in logged_models:
        if lm.name == model_name_pattern:
            if matching_model is None or lm.creation_timestamp > matching_model.creation_timestamp:
                matching_model = lm

    if matching_model:
        try:
            if "xgboost" in model_type:
                m = mlflow.xgboost.load_model(matching_model.model_uri)
                dmatrix = xgb.DMatrix(X_holdout, feature_names=list(X_holdout.columns))
                yp = m.predict(dmatrix)
            else:
                m = mlflow.sklearn.load_model(matching_model.model_uri)
                yp = m.predict_proba(X_holdout)[:, 1]

            y_p = (yp > 0.5).astype(int)
            loaded_models[display_name] = m
            model_predictions[display_name] = {"y_pred": y_p, "y_proba": yp}
            print(f"Loaded {display_name}: ROC-AUC = {roc_auc_score(y_actual, yp):.4f}")
        except Exception as e:
            print(f"Could not load {display_name}: {e}")

print(f"\nLoaded {len(loaded_models)} models for comparison")

In [ ]:
# Model Comparison Grid (3 columns x 3 rows)
n_models = len(model_predictions)
if n_models > 0:
    fig, axes = plt.subplots(3, n_models, figsize=(5 * n_models, 12))
    if n_models == 1:
        axes = axes.reshape(-1, 1)

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

    for col_idx, (name, preds) in enumerate(model_predictions.items()):
        y_p = preds["y_pred"]
        yp = preds["y_proba"]
        color = colors[col_idx % len(colors)]

        # Row 1: Confusion Matrix
        cm = confusion_matrix(y_actual, y_p)
        ax = axes[0, col_idx]
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["Pred 0", "Pred 1"])
        ax.set_yticklabels(["Actual 0", "Actual 1"])
        for i in range(2):
            for j in range(2):
                pct = cm[i, j] / cm.sum() * 100
                ax.text(j, i, f"{cm[i, j]}\n({pct:.1f}%)", ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=10)
        acc = accuracy_score(y_actual, y_p)
        ax.set_title(f"{name}\nAccuracy: {acc:.3f}", fontsize=11, fontweight="bold")

        # Row 2: ROC Curve
        ax = axes[1, col_idx]
        fpr, tpr, _ = roc_curve(y_actual, yp)
        auc = roc_auc_score(y_actual, yp)
        ax.plot(fpr, tpr, color=color, lw=2, label=f"AUC = {auc:.4f}")
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
        ax.fill_between(fpr, tpr, alpha=0.2, color=color)
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curve", fontsize=10)
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)

        # Row 3: Precision-Recall Curve
        ax = axes[2, col_idx]
        precision_vals, recall_vals, _ = precision_recall_curve(y_actual, yp)
        pr_auc = average_precision_score(y_actual, yp)
        ax.plot(recall_vals, precision_vals, color=color, lw=2, label=f"PR-AUC = {pr_auc:.4f}")
        baseline = y_actual.sum() / len(y_actual)
        ax.axhline(y=baseline, color="gray", linestyle="--", lw=1, label=f"Baseline = {baseline:.2f}")
        ax.fill_between(recall_vals, precision_vals, alpha=0.2, color=color)
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title("Precision-Recall Curve", fontsize=10)
        ax.legend(loc="lower left")
        ax.grid(True, alpha=0.3)

    plt.suptitle("Model Comparison Grid: Holdout Set Performance",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No models loaded for comparison")

In [ ]:
# Summary metrics table for all models
if model_predictions:
    comparison_results = []
    for name, preds in model_predictions.items():
        y_p = preds["y_pred"]
        yp = preds["y_proba"]
        comparison_results.append({
            "Model": name,
            "ROC-AUC": roc_auc_score(y_actual, yp),
            "PR-AUC": average_precision_score(y_actual, yp),
            "F1-Score": f1_score(y_actual, y_p),
            "Precision": precision_score(y_actual, y_p, zero_division=0),
            "Recall": recall_score(y_actual, y_p, zero_division=0),
            "Accuracy": accuracy_score(y_actual, y_p),
        })

    comparison_df = pd.DataFrame(comparison_results).set_index("Model")
    print("\n" + "=" * 70)
    print("MODEL COMPARISON SUMMARY (Holdout Set)")
    print("=" * 70)
    display(
        comparison_df.style
        .highlight_max(axis=0, props="background-color: #2e7d32; color: white")
        .format("{:.4f}")
    )

    best_model_name = comparison_df["ROC-AUC"].idxmax()
    best_auc = comparison_df.loc[best_model_name, "ROC-AUC"]
    print(f"\nBest Model: {best_model_name} (ROC-AUC = {best_auc:.4f})")

## 11.4 Adversarial Pipeline Validation

Validate that scoring pipeline produces identical features to training for holdout entities.
This catches transformation inconsistencies (e.g., scalers re-fit, encoders handling unseen values differently).

In [ ]:
gold_features = pd.read_parquet(get_gold_path())

holdout_mask = gold_features[ORIGINAL_COLUMN].notna()
holdout_gold = gold_features[holdout_mask].copy()
print(f"Holdout entities for validation: {holdout_mask.sum():,}")

# Compare scoring features vs gold features for holdout records
scoring_entity_ids = set(scoring_features[FEAST_ENTITY_KEY].values)
gold_holdout = holdout_gold[holdout_gold[FEAST_ENTITY_KEY].isin(scoring_entity_ids)]

exclude_cols = {FEAST_ENTITY_KEY, "event_timestamp", TARGET_COLUMN, ORIGINAL_COLUMN}
compare_cols = [
    c for c in gold_holdout.columns
    if c not in exclude_cols and not c.startswith("original_")
]

print("\n" + "=" * 60)
print("ADVERSARIAL PIPELINE VALIDATION")
print("=" * 60)

mismatches = []
for col in compare_cols:
    if col in scoring_features.columns and col in gold_holdout.columns:
        g_vals = gold_holdout[col].values
        s_vals = scoring_features.reindex(gold_holdout.index)[col].values
        if pd.api.types.is_numeric_dtype(gold_holdout[col]):
            delta = np.abs(g_vals.astype(float) - s_vals.astype(float))
            max_delta = np.nanmax(delta) if len(delta) > 0 else 0
            if max_delta > 1e-6:
                mismatches.append({"feature": col, "max_delta": max_delta})

if not mismatches:
    print("\nPASSED: Scoring features match training features")
else:
    print(f"\nFAILED: {len(mismatches)} features with drift")
    display(pd.DataFrame(mismatches).sort_values("max_delta", ascending=False))

## 11.5 Transformation Validation

Use `validate_feature_transformation()` from the validation module to verify
encoding/scaling consistency between training and scoring.

In [ ]:
from customer_retention.stages.validation import validate_feature_transformation

# Training features = non-holdout, scoring features = holdout
training_mask = gold_features[ORIGINAL_COLUMN].isna()
training_subset = gold_features[training_mask].copy()
scoring_subset = gold_features[~training_mask].copy()

report = validate_feature_transformation(
    training_df=training_subset,
    scoring_df=scoring_subset,
    transform_fn=prepare_features,
    entity_column=FEAST_ENTITY_KEY,
    verbose=True,
)

if report.passed:
    print("Transformation validation PASSED")
else:
    print(f"Transformation validation FAILED: {len(report.feature_mismatches)} mismatches")

## 11.6 Model Explanations (SHAP)

In [ ]:
import shap

# Load best model for SHAP
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()

experiment = client.get_experiment_by_name(PIPELINE_NAME)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.best_roc_auc DESC"],
    max_results=1,
)
parent_run = runs[0]

best_model_tag = parent_run.data.tags.get("best_model", "random_forest")
model_name = f"model_{best_model_tag}"
if RECOMMENDATIONS_HASH:
    model_name = f"{model_name}_{RECOMMENDATIONS_HASH}"

child_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'",
)
model_run = next((c for c in child_runs if c.info.run_name == best_model_tag), parent_run)

model_uri = f"runs:/{model_run.info.run_id}/{model_name}"
print(f"Loading model: {model_uri}")
if best_model_tag == "xgboost":
    model = mlflow.xgboost.load_model(model_uri)
else:
    model = mlflow.sklearn.load_model(model_uri)
print(f"Model type: {type(model).__name__}")

In [ ]:
# Prepare features for SHAP using TransformExecutor (NOT LabelEncoder)
X = prepare_features(scoring_features)
feature_names = list(X.columns)
print(f"Prepared {len(feature_names)} features for SHAP analysis")

In [ ]:
# Create SHAP explainer
print("Creating SHAP explainer (may take a moment)...")

background_size = min(100, len(X))
background = shap.sample(X, background_size)

if hasattr(model, "predict_proba"):
    explainer = shap.Explainer(model.predict_proba, background, feature_names=feature_names)
else:
    explainer = shap.Explainer(model, background, feature_names=feature_names)

print("Computing SHAP values...")
shap_values = explainer(X)
print(f"SHAP values computed for {len(shap_values)} records")

In [ ]:
# Use positive class SHAP values if multi-output
if len(shap_values.shape) == 3:
    shap_vals = shap_values[:, :, 1]  # Positive class
else:
    shap_vals = shap_values

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X, feature_names=feature_names, show=False, max_display=20)
plt.title("Feature Importance (SHAP Summary)")
plt.tight_layout()
plt.show()

In [ ]:
# Mean absolute SHAP values
mean_shap = np.abs(shap_vals.values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": mean_shap,
}).sort_values("importance", ascending=False)

print("Top 15 Most Important Features:")
display(importance_df.head(15))

## 11.7 Customer Browser

In [ ]:
# Create combined dataset for browsing
browser_df = predictions_df.merge(
    scoring_features[[FEAST_ENTITY_KEY] + feature_names],
    on=FEAST_ENTITY_KEY,
    how="left",
)

print(f"Customer browser ready with {len(browser_df):,} records")
print(f"\nPrediction Distribution:")
print(f"  Predicted Positive: {(browser_df['prediction'] == 1).sum():,}")
print(f"  Predicted Negative: {(browser_df['prediction'] == 0).sum():,}")
print(f"\nCorrect Predictions: {browser_df['correct'].sum():,}/{len(browser_df):,} ({browser_df['correct'].mean():.1%})")

In [ ]:
def show_customer(idx: int):
    """Display details and SHAP explanation for a single customer."""
    row = browser_df.iloc[idx]
    entity_id = row[FEAST_ENTITY_KEY]

    print(f"=== Customer {entity_id} ===")
    print(f"Prediction: {int(row['prediction'])} (probability: {row['probability']:.3f})")
    print(f"Actual: {int(row['actual'])}")
    print(f"Correct: {'Yes' if row['correct'] else 'No'}")
    print()

    feature_vals = X.iloc[idx]
    if len(shap_values.shape) == 3:
        customer_shap = shap_values[idx, :, 1].values
    else:
        customer_shap = shap_values[idx].values

    feature_impact = pd.DataFrame({
        "feature": feature_names,
        "value": feature_vals.values,
        "shap_impact": customer_shap,
    }).sort_values("shap_impact", key=abs, ascending=False)

    print("Top Contributing Features:")
    display(feature_impact.head(10))

    # Waterfall plot
    plt.figure(figsize=(10, 6))
    if len(shap_values.shape) == 3:
        shap.plots.waterfall(shap_values[idx, :, 1], max_display=10, show=False)
    else:
        shap.plots.waterfall(shap_values[idx], max_display=10, show=False)
    plt.title(f"SHAP Explanation for Customer {entity_id}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Show first 3 customers
print("Showing first 3 customers:\n")
for i in range(min(3, len(browser_df))):
    show_customer(i)
    print("\n" + "=" * 60 + "\n")

In [ ]:
# Look up by entity ID
def lookup_customer(entity_id):
    """Find and display a customer by their entity ID."""
    mask = browser_df[FEAST_ENTITY_KEY] == entity_id
    if not mask.any():
        print(f"Customer {entity_id} not found in scoring set")
        return
    idx = browser_df[mask].index[0]
    x_idx = browser_df.index.get_loc(idx)
    show_customer(x_idx)


# Example: lookup_customer(12345)
print("Available entity IDs (first 10):")
print(browser_df[FEAST_ENTITY_KEY].head(10).tolist())

## 11.8 Error Analysis

In [ ]:
# Analyze misclassified customers
incorrect = browser_df[browser_df["correct"] == 0]
print(f"Misclassified customers: {len(incorrect):,}")

# False positives (predicted 1, actual 0)
fp = incorrect[incorrect["prediction"] == 1]
print(f"  False Positives: {len(fp):,}")

# False negatives (predicted 0, actual 1)
fn = incorrect[incorrect["prediction"] == 0]
print(f"  False Negatives: {len(fn):,}")

In [ ]:
# Example false positive
if len(fp) > 0:
    print("\n=== Example False Positive ===")
    fp_idx = browser_df.index.get_loc(fp.index[0])
    show_customer(fp_idx)

In [ ]:
# Example false negative
if len(fn) > 0:
    print("\n=== Example False Negative ===")
    fn_idx = browser_df.index.get_loc(fn.index[0])
    show_customer(fn_idx)

## 11.9 Export Results

In [ ]:
# Export detailed results with feature importance
output_dir = GEN_EXPERIMENTS_DIR / "data" / "scoring"
output_dir.mkdir(parents=True, exist_ok=True)

# Save global feature importance
importance_df.to_csv(output_dir / "feature_importance.csv", index=False)
print(f"Feature importance saved to {output_dir / 'feature_importance.csv'}")

top_features = importance_df.head(10)["feature"].tolist()
shap_by_entity = pd.DataFrame({FEAST_ENTITY_KEY: scoring_features[FEAST_ENTITY_KEY].values})
for feat in top_features:
    feat_idx = feature_names.index(feat)
    if len(shap_values.shape) == 3:
        shap_by_entity[f"shap_{feat}"] = shap_values[:, feat_idx, 1].values
    else:
        shap_by_entity[f"shap_{feat}"] = shap_values[:, feat_idx].values

detailed_df = predictions_df.merge(shap_by_entity, on=FEAST_ENTITY_KEY, how="left")
detailed_df.to_parquet(output_dir / "predictions_with_shap.parquet", index=False)
print(f"Detailed predictions with SHAP saved to {output_dir / 'predictions_with_shap.parquet'}")


> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.